In [56]:
import pandas as pd
import pickle
import sys
from nltk.tokenize import RegexpTokenizer
from sklearn.feature_extraction.text import CountVectorizer
sys.path.append("../new-2")   # path to directory containing detector.py
from sklearn.metrics import accuracy_score,classification_report

from detector import SpamMessageDetector

In [2]:
test_df = pd.read_csv("dataset\CEAS_08.csv")
test_df

<>:1: SyntaxWarning: invalid escape sequence '\C'
<>:1: SyntaxWarning: invalid escape sequence '\C'
C:\Users\ASUS\AppData\Local\Temp\ipykernel_21532\3004789923.py:1: SyntaxWarning: invalid escape sequence '\C'
  test_df = pd.read_csv("dataset\CEAS_08.csv")


,sender,receiver,date,subject,body,label,urls
0,Young Esposito <Young@iworld.de>,user4@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 16:31:02 -0700",Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",1,1
1,Mok <ipline's1983@icable.ph>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 18:31:03 -0500",Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,1,1
2,Daily Top 10 <Karmandeep-opengevl@universalnet...,user2.9@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 20:28:00 -1200",CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1,1
3,Michael Parker <ivqrnai@pobox.com>,SpamAssassin Dev <xrh@spamassassin.apache.org>,"Tue, 05 Aug 2008 17:31:20 -0600",Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,0,1
4,Gretchen Suggs <externalsep1@loanofficertool.com>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 19:31:21 -0400",SpecialPricesPharmMoreinfo,\nWelcomeFastShippingCustomerSupport\nhttp://7...,1,1
...,...,...,...,...,...,...,...
39149,CNN Alerts <charlene-detecton@btcmarketing.com>,email1007@gvc.ceas-challenge.cc,"Fri, 08 Aug 2008 10:34:50 -0400",CNN Alerts: My Custom Alert,\n\nCNN Alerts: My Custom Alert\n\n\n\n\n\n\n ...,1,0
39150,CNN Alerts <idgetily1971@careplusnj.org>,email104@gvc.ceas-challenge.cc,"Fri, 08 Aug 2008 10:35:11 -0400",CNN Alerts: My Custom Alert,\n\nCNN Alerts: My Custom Alert\n\n\n\n\n\n\n ...,1,0
39151,Abhijit Vyas <xpojhbz@gmail.com>,fxgmqwjn@triptracker.net,"Fri, 08 Aug 2008 22:00:43 +0800",Slideshow viewer,Hello there ! \nGreat work on the slide show v...,0,0
39152,Joseph Brennan <vupzesm@columbia.edu>,zqoqi@spamassassin.apache.org,"Fri, 08 Aug 2008 09:00:46 -0500",Note on 2-digit years,"\nMail from sender , coming from intuit.com\ns...",0,0


old methods

In [42]:
old_test_df =pd.read_csv("dataset\\old.csv")
old_test_df

,Unnamed: 0,text,label_num,label
0,0,"Buck up, your troubles caused by small dimensi...",1,spam
1,1,\nUpgrade your sex and pleasures with these te...,1,spam
2,2,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1,spam
3,3,Would anyone object to removing .so from this ...,0,ham
4,4,\nWelcomeFastShippingCustomerSupport\nhttp://7...,1,spam
...,...,...,...,...
39149,39149,\n\nCNN Alerts: My Custom Alert\n\n\n\n\n\n\n ...,1,spam
39150,39150,\n\nCNN Alerts: My Custom Alert\n\n\n\n\n\n\n ...,1,spam
39151,39151,Hello there ! \nGreat work on the slide show v...,0,ham
39152,39152,"\nMail from sender , coming from intuit.com\ns...",0,ham


In [43]:
def clean_str(string, reg = RegexpTokenizer(r'[a-z]+')):
    string = string.lower()
    tokens = reg.tokenize(string)
    return " ".join(tokens)

In [44]:
old_test_df['text'] = old_test_df['text'].apply(lambda string: clean_str(string))

In [45]:
def rem_sub(i):
    return i.replace('subject', '')
old_test_df['text'] = old_test_df['text'].map(rem_sub)

In [49]:
x = old_test_df['text']
Y = old_test_df.label_num

In [50]:
cv = pickle.load(open("models/vectorizer.pkl", "rb"))
old_test_x = cv.transform(x)

# Get the categories
old_test_y = old_test_df.label

In [51]:
print("X Shape:",old_test_x.shape)
# print("y Shape:",x_test.shape)

X Shape: (39154, 45740)


In [28]:
model_files = {
    "DecisionTree": "models\\decision_tree.pkl",
    "LogisticRegression": "models\\logistic_regression.pkl",
    "MultinomialNB": "models\\multinomialNb.pkl",
    "RandomForest": "models\\random_forest.pkl"
}

loaded_models = {}

# Load each model
for name, file_path in model_files.items():
    with open(file_path, "rb") as f:
        loaded_models[name] = pickle.load(f)
        print(f"Loaded: {name} from {file_path}")

# Now you can access individual models:
dt_model = loaded_models["DecisionTree"]
lr_model = loaded_models["LogisticRegression"]
nb_model = loaded_models["MultinomialNB"]
rf_model = loaded_models["RandomForest"]



Loaded: DecisionTree from models\decision_tree.pkl
Loaded: LogisticRegression from models\logistic_regression.pkl
Loaded: MultinomialNB from models\multinomialNb.pkl
Loaded: RandomForest from models\random_forest.pkl


In [58]:
dt_prediction = dt_model.predict(old_test_x)
accuracy = accuracy_score(dt_prediction,old_test_y)
print(accuracy)



0.6539561730602237


In [ ]:
lr_prediction = lr_model.predict(old_test_x)
accuracy = accuracy_score(lr_prediction,old_test_y)
print(accuracy)

0.7951933391224396


In [60]:
nb_prediction = nb_model.predict(old_test_x)
accuracy = accuracy_score(nb_prediction,old_test_y)
print(accuracy)

0.7018440006129641


In [ ]:
rf_prediction = rf_model.predict(old_test_x)
accuracy = accuracy_score(rf_prediction,old_test_y)
print(accuracy)

new method

In [11]:
new_test_df = pd.read_csv("dataset\\new.csv")
new_test_df

,Unnamed: 0,text,label
0,0,"Buck up, your troubles caused by small dimensi...",spam
1,1,\nUpgrade your sex and pleasures with these te...,spam
2,2,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,spam
3,3,Would anyone object to removing .so from this ...,ham
4,4,\nWelcomeFastShippingCustomerSupport\nhttp://7...,spam
...,...,...,...
39149,39149,\n\nCNN Alerts: My Custom Alert\n\n\n\n\n\n\n ...,spam
39150,39150,\n\nCNN Alerts: My Custom Alert\n\n\n\n\n\n\n ...,spam
39151,39151,Hello there ! \nGreat work on the slide show v...,ham
39152,39152,"\nMail from sender , coming from intuit.com\ns...",ham


In [12]:
spam_detector = SpamMessageDetector("mshenoda/roberta-spam")
spam_detector.evaluate("dataset/new.csv")

KeyboardInterrupt: 